In [ ]:
%pip install rotary-embedding-torch

#train_pos = 40248000
#val_pos  = 402480
#def get_batch(split):
#    global train_pos, val_pos

#    data = train_data if split == 'train' else val_data
#    pos = train_pos if split == 'train' else val_pos

#    x_list, y_list = [], []

#    for _ in range(batch):   
#        if pos + seq_len + 1 > len(data):  
#            pos = 0

#        x = data[pos : pos + seq_len]
#        y = data[pos + 1 : pos + seq_len + 1]

#        x_list.append(torch.tensor(x, dtype=torch.long))
#        y_list.append(torch.tensor(y, dtype=torch.long))

#        pos += 258

   
#    if split == 'train':
#        train_pos = pos
#    else:
#        val_pos = pos

#    perm = torch.randperm(batch)

#    x_batch = torch.stack(x_list)[perm].to(device)
#    y_batch = torch.stack(y_list)[perm].to(device)
#    return x_batch, y_batch

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from rotary_embedding_torch import RotaryEmbedding
from transformers import GPT2TokenizerFast
import torch
import torch.nn as nn
import torch.functional as F
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seq_len = 512
batch = 16
# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vocab_size = len(tokenizer)
print(f"Vocabulary size: {vocab_size}")

def encode(text):
    """Encode text using GPT-2 tokenizer"""
    return tokenizer.encode(text, add_special_tokens=False)

def decode(indices):
    """Decode token indices back to text"""
    return tokenizer.decode(indices)

: 

In [ ]:
#7500 more steps
import numpy as np
import os


chunk_size = 1_000_000  # characters
tokens = []
if not os.path.exists("/content/tokens.npy"):
  with open("/content/drive/MyDrive/Combined.txt", "r", encoding="utf-8") as f:
      while True:
          chunk = f.read(chunk_size)
          if not chunk:
              break
          tokens.extend(encode(chunk))

  tokens = np.array(tokens, dtype=np.int32)
  invalid = np.where((tokens < 0) | (tokens >= vocab_size))[0]
  print(f"Found {len(invalid)} invalid tokens at positions {invalid[:10]}...")
  np.save("tokens.npy", tokens)
  tokens = np.memmap("tokens.npy", dtype=np.int32, mode="r")
else:
   tokens = np.memmap("tokens.npy", dtype=np.int32, mode="r")
n = int(0.9*len(tokens))
train_data = tokens[:n]
val_data = tokens[n:]
def get_batch(split):
    data = train_data if split == 'train' else val_data
    max_start = len(data) - seq_len 
    ix = torch.randint(0,max_start , (batch,))

    x = torch.stack([
        torch.from_numpy(data[i:i+seq_len].astype(np.int64))
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(data[i+1:i+seq_len+1].astype(np.int64))
        for i in ix
    ])

    x = x.to(device)
    y = y.to(device)
    return x, y




In [2]:
d_model = 512
heads = 8
n_layers = 8

class causal_self_attention(nn.Module):
  def __init__(self,d_model,Heads):
    super().__init__()
    assert d_model % Heads == 0, "d_model must be divisible by n_head"
    self.d_model = d_model
    self.n_head = Heads
    self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
    self.c_proj = nn.Linear(d_model, d_model, bias=False)
    self.head_dim = d_model // Heads
    self.rotary = RotaryEmbedding(dim=self.head_dim)
  def forward(self,x):
    B,T,C = x.size()
    q, k, v  = self.c_attn(x).split(self.d_model, dim=2)
    k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
    q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
    v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
    q = self.rotary.rotate_queries_or_keys(q)
    k = self.rotary.rotate_queries_or_keys(k)
    y = torch.nn.functional.scaled_dot_product_attention(q,k,v,is_causal=True)
    y = y.transpose(1, 2).contiguous().view(B, T, C)
    y = self.c_proj(y)
    return y


class TransformerBlock(nn.Module):
  def __init__(self,d_model, heads):
    super().__init__()
    self.norm1 = nn.RMSNorm(d_model)
    self.att = causal_self_attention(d_model,heads)
    self.norm2 = nn.RMSNorm(d_model)
    self.mlp = nn.Sequential(
        nn.Linear(d_model,d_model*4),
        nn.GELU(),
        nn.Linear(d_model*4,d_model),
    )
  def forward(self,x):
   h = self.norm1(x)
   x = x + self.att(h)
   x = x + self.mlp(self.norm2(x))
   return x
class Transformer(nn.Module):
    def __init__(self, d_model, heads, n_layers):
        super().__init__()

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, heads)
            for _ in range(n_layers)
        ])

        self.norm = nn.RMSNorm(d_model)

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.norm(x)
class GPT(nn.Module):
  def __init__(self,vocab_size, d_model, heads, n_layers):
     super().__init__()
     self.token_emb = nn.Embedding(vocab_size,d_model)
     self.transformer = Transformer(d_model, heads, n_layers)
     self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
     self.lm_head.weight = self.token_emb.weight

  def forward(self, idx):

        # Token + position embeddings
        x = self.token_emb(idx)                 # (batch, seq_len, d_model)
        x = self.transformer(x)                 # (batch, seq_len, d_model)

        # Project to vocab
        logits = self.lm_head(x)                # (batch, seq_len, vocab_size)

        return logits
model = GPT(vocab_size, d_model, heads, n_layers)


def init_weights(module):
    if isinstance(module, nn.Linear):
        # Standard Xavier initialization for linear layers
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        # Small normal initialization for embeddings
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, nn.LayerNorm) or isinstance(module, nn.RMSNorm):
        # LayerNorm weights to 1, bias to 0
        nn.init.ones_(module.weight)

# Apply to the model
model.apply(init_weights)
m = model.to(device)
checkpoint = torch.load(r"C:\Users\conta\Downloads\model.pth",map_location=torch.device("cpu"))# remove this if not on a cpu
# Remove the '_orig_mod.' prefix if present
state_dict = {k.replace("_orig_mod.", ""): v for k, v in checkpoint.items()}
# Load into your normal model
m.load_state_dict(state_dict)
optimizer = torch.optim.AdamW(
    m.parameters(),
    lr=1e-4,
    weight_decay=0.1,
    betas=(0.9,0.95),
    fused=True
)

#m = torch.compile(m)


In [ ]:
criterion = nn.CrossEntropyLoss()

def train_step():
    m.train()

    x, y = get_batch('train')           # x,y: (B, T)

    logits = m(x)                       # (B, T, vocab)
    B, T, V = logits.shape

    loss = criterion(
        logits.view(B*T, V),
        y.view(B*T)
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    optimizer.step()

    return loss.item()
@torch.no_grad()
def eval_step():
    m.eval()

    x, y = get_batch('val')
    logits = m(x)

    B, T, V = logits.shape
    loss = criterion(
        logits.view(B*T, V),
        y.view(B*T)
    )
    return loss.item()
max_iters = 1000
eval_interval = 100

for step in range(max_iters):
    loss = train_step()

    if step % 10 == 0:
        print(f"step {step} | train loss {loss:.4f}")

    if step % eval_interval == 0:
        val_loss = eval_step()
        print(f"--- val loss {val_loss:.4f} ---")


step 0 | train loss 3.3425
--- val loss 3.4288 ---
step 10 | train loss 3.0942
step 20 | train loss 2.9205
step 30 | train loss 3.2701
step 40 | train loss 3.6107
step 50 | train loss 3.0178
step 60 | train loss 3.2191
step 70 | train loss 3.1587
step 80 | train loss 3.1475
step 90 | train loss 3.5090
step 100 | train loss 2.8803
--- val loss 3.3526 ---
step 110 | train loss 2.9855
step 120 | train loss 2.9519
step 130 | train loss 3.3552
step 140 | train loss 3.0474
step 150 | train loss 3.3307
step 160 | train loss 3.0810
step 170 | train loss 3.1118
step 180 | train loss 3.2383
step 190 | train loss 3.3358
step 200 | train loss 2.7866
--- val loss 3.4839 ---
step 210 | train loss 3.3016
step 220 | train loss 3.2690
step 230 | train loss 3.0680
step 240 | train loss 3.1391
step 250 | train loss 2.8695
step 260 | train loss 3.2600
step 270 | train loss 3.5915
step 280 | train loss 2.8864
step 290 | train loss 2.8320
step 300 | train loss 3.2938
--- val loss 2.9599 ---
step 310 | train

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [4]:
import torch.nn.functional as F
import time

@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.5, top_k=None):
    model.eval()
    idx = idx.to(next(model.parameters()).device)  # move to same device as model
    for _ in range(max_new_tokens):
        # crop idx to last `seq_len` tokens to fit model input
        idx_cond = idx[:, -seq_len:]
        # forward pass
        logits = m(idx_cond)              # (1, seq_len, vocab_size)
        logits = logits[:, -1, :]             # (1, vocab) -> last token

        # apply temperature
        logits = logits / temperature

        # optionally restrict to top_k
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

        # convert to probabilities
        probs = F.softmax(logits, dim=-1)

        # sample next token
        next_idx = torch.multinomial(probs, num_samples=1)
        # append to sequence
        idx = torch.cat([idx, next_idx], dim=1)
    return idx

# starting token(s)
start_text = "He stood there"
start_tokens = torch.tensor([encode(start_text)], dtype=torch.long)

# generate 50 new tokens
generated_tokens = generate(m, start_tokens, max_new_tokens=100, temperature=0.7, top_k=50)

generated_text = decode(generated_tokens[0].tolist())

words = generated_text.split(" ")
for w in words:
    print(w + " ", end="", flush=True)
    time.sleep(0.02 if w != "\n" else 0.2)

He stood there a moment longer before he could speak again.
“I am so sorry.” He stuttered. “Did something happen?”
“Dude.”
“No. I was just… I was trying to get to the tower so I could help.”
“You were never going to tell me.”
“Not when you didn’t mean to.”
“I should.”
 